# 🎬 YouTube ETL & Sentiment Analysis Platform

**Professional-grade data pipeline for YouTube analytics with sentiment analysis**

## 📋 Notebook Overview

This is the **primary ETL notebook** for the YouTube analytics platform. It provides:

- **Complete YouTube Data ETL Pipeline** - Fetch videos, metrics, and comments from multiple channels
- **Advanced Sentiment Analysis** - VADER-based sentiment scoring with channel grouping
- **Interactive Data Visualization** - Professional Plotly charts with weighted popularity algorithms  
- **Data Quality Monitoring** - Comprehensive diagnostics and ETL health checks
- **Professional Text Processing** - Clean channel names and video titles for analysis

## 🎯 Quick Start

1. **Setup Environment**: 
   - Copy `.env.example` to `.env`
   - Add your YouTube API key and database credentials
   - Configure channel URLs for the creators you want to analyze

2. **Run Database Bootstrap**: Execute the first cell to create required tables

3. **Configure Channels**: 
   - Set environment variables like `YT_CHANNEL_1_YT=https://youtube.com/@yourchannel`
   - Add as many channels as needed (YT_CHANNEL_2_YT, YT_CHANNEL_3_YT, etc.)

4. **Execute ETL Pipeline**: Run multi-channel ETL cells to fetch all data

5. **Perform Sentiment Analysis**: Process comments for sentiment scoring

6. **Generate Visualizations**: Create channel popularity and sentiment dashboards

## 📊 Expected Output

- **Videos from your configured channels** - Automatically discovers all uploads
- **Complete comment dataset** with full sentiment analysis coverage
- **Interactive charts** showing channel popularity trends and sentiment distributions  
- **Data quality metrics** with automated health monitoring

## 🔧 Configuration

The platform is designed to work with **any YouTube channels**. Simply configure your `.env` file with:

```bash
# YouTube API Configuration
YOUTUBE_API_KEY=your_api_key_here

# Database Configuration  
DB_HOST=localhost
DB_PORT=3306
DB_USER=your_db_user
DB_PASS=your_db_password
DB_NAME=your_database_name

# Channel Configuration (add as many as needed)
YT_CHANNEL_1_YT=https://youtube.com/@channel1
YT_CHANNEL_2_YT=https://youtube.com/@channel2
YT_CHANNEL_3_YT=https://youtube.com/@channel3
```

---

In [ ]:
# DB bootstrap: create database and minimal tables if missing
import os, pymysql
from dotenv import load_dotenv
load_dotenv()

host=os.getenv('DB_HOST','127.0.0.1')
port=int(os.getenv('DB_PORT','3306'))
user=os.getenv('DB_USER')
pw=os.getenv('DB_PASS') or ''
db=os.getenv('DB_NAME') or os.getenv('DB_NAME_PRIVATE')
print({'host':host,'port':port,'db':db})

# Create database if not exists
conn = pymysql.connect(host=host, port=port, user=user, password=pw)
try:
    with conn.cursor() as cur:
        cur.execute(f"CREATE DATABASE IF NOT EXISTS `{db}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
    conn.commit()
finally:
    conn.close()

# Create required tables if not exist
conn = pymysql.connect(host=host, port=port, user=user, password=pw, db=db, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)
try:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_videos_raw` (
              `video_id` varchar(50) NOT NULL,
              `playlist_id` varchar(100) DEFAULT NULL,
              `raw_data` json DEFAULT NULL,
              `fetched_at` datetime DEFAULT CURRENT_TIMESTAMP,
              `processed` tinyint(1) DEFAULT '0',
              PRIMARY KEY (`video_id`),
              KEY `idx_yvraw_playlist` (`playlist_id`),
              KEY `idx_yvraw_processed` (`processed`)
            )
            """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_metrics` (
              `video_id` varchar(50) NOT NULL,
              `view_count` bigint DEFAULT NULL,
              `like_count` bigint DEFAULT NULL,
              `dislike_count` bigint DEFAULT NULL,
              `comment_count` bigint DEFAULT NULL,
              `subscriber_count` bigint DEFAULT NULL,
              `metrics_date` date NOT NULL,
              `fetched_at` datetime NOT NULL DEFAULT CURRENT_TIMESTAMP,
              PRIMARY KEY (`video_id`,`metrics_date`)
            )
            """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_etl_runs` (
              `channel_id` varchar(64) NOT NULL,
              `run_date` date NOT NULL,
              `started_at` datetime NOT NULL DEFAULT CURRENT_TIMESTAMP,
              `finished_at` datetime DEFAULT NULL,
              `status` varchar(16) NOT NULL DEFAULT 'started',
              PRIMARY KEY (`channel_id`,`run_date`)
            )
            """
        )
    conn.commit()
    with conn.cursor() as cur:
        cur.execute("SHOW TABLES LIKE 'youtube_%'")
        tables=[row[next(iter(row))] for row in cur.fetchall()]
        print({'tables': tables})
finally:
    conn.close()

# YouTube ETL: fetch channel videos and store raw + daily-max metrics

This section fetches all upload videos for configured channels using YouTube Data API v3,
stores the raw video JSON into `youtube_videos_raw`, and upserts a daily max row into `youtube_metrics`.

Set your `YOUTUBE_API_KEY` and channel vars in `.env` before running.

In [ ]:
# Imports and helpers
import os
import json
import re
from datetime import datetime, date
import requests
import pymysql
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv

load_dotenv()
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = int(os.getenv('DB_PORT', '3306'))
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_NAME = os.getenv('DB_NAME')

assert YOUTUBE_API_KEY, 'YOUTUBE_API_KEY not set in .env'
assert DB_NAME, 'DB_NAME not set in .env'


def get_channel_id_from_url(url):
    # Accepts channel URLs like /channel/UC..., /user/..., or /@handle and resolves to channelId via API when needed
    url = url.strip()
    if url.endswith('/'):
        url = url[:-1]
    parsed = urlparse(url)
    path = parsed.path.lstrip('/')
    # direct channel id
    m = re.match(r'channel/(UC[0-9A-Za-z_-]{20,})', path)
    if m:
        return m.group(1)
    # handle or user or @handle
    candidate = path.split('/')[-1]
    if candidate:
        # call search.list to resolve handle/user to a channelId
        url_api = 'https://www.googleapis.com/youtube/v3/search'
        params = {'key': YOUTUBE_API_KEY, 'q': candidate, 'type': 'channel', 'part': 'snippet', 'maxResults': 5}
        r = requests.get(url_api, params=params)
        data = r.json()
        items = data.get('items', [])
        if items:
            return items[0]['snippet']['channelId']
    return None


def get_uploads_playlist_for_channel(channel_id):
    # channels.list part=contentDetails to get uploads playlist id
    url_api = 'https://www.googleapis.com/youtube/v3/channels'
    params = {'key': YOUTUBE_API_KEY, 'id': channel_id, 'part': 'contentDetails'}
    r = requests.get(url_api, params=params)
    data = r.json()
    items = data.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists'].get('uploads')


def list_playlist_videos(playlist_id):
    # yields playlistItem per page
    url_api = 'https://www.googleapis.com/youtube/v3/playlistItems'
    params = {'key': YOUTUBE_API_KEY, 'playlistId': playlist_id, 'part': 'contentDetails,snippet', 'maxResults': 50}
    while True:
        r = requests.get(url_api, params=params)
        data = r.json()
        for it in data.get('items', []):
            yield it
        if 'nextPageToken' in data:
            params['pageToken'] = data['nextPageToken']
        else:
            break


def get_videos_details(video_ids):
    # video_ids up to 50
    url_api = 'https://www.googleapis.com/youtube/v3/videos'
    params = {'key': YOUTUBE_API_KEY, 'id': ','.join(video_ids), 'part': 'snippet,contentDetails,statistics'}
    r = requests.get(url_api, params=params)
    return r.json()


def connect_db():
    return pymysql.connect(host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASS, db=DB_NAME, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)


def upsert_video_raw(conn, video_id, playlist_id, raw_json):
    with conn.cursor() as cur:
        sql = "INSERT INTO youtube_videos_raw (video_id, playlist_id, raw_data, fetched_at, processed) VALUES (%s, %s, %s, NOW(), 0) ON DUPLICATE KEY UPDATE raw_data = VALUES(raw_data), fetched_at = NOW(), processed = 0"
        cur.execute(sql, (video_id, playlist_id, json.dumps(raw_json)))
    conn.commit()


def upsert_daily_metrics(conn, video_id, view_count, like_count, comment_count, fetched_dt=None):
    # maintain one max-per-day row: if existing for (video_id, today) has lower view_count, replace it
    if fetched_dt is None:
        fetched_dt = datetime.utcnow()
    metrics_date = fetched_dt.date()
    with conn.cursor() as cur:
        # Try insert; on duplicate, update only if view_count is greater
        sql = "INSERT INTO youtube_metrics (video_id, view_count, like_count, dislike_count, comment_count, subscriber_count, metrics_date, fetched_at) VALUES (%s,%s,%s,%s,%s,NULL,%s,NOW()) ON DUPLICATE KEY UPDATE view_count = IF(VALUES(view_count) > view_count, VALUES(view_count), view_count), like_count = IF(VALUES(like_count) > like_count, VALUES(like_count), like_count), comment_count = IF(VALUES(comment_count) > comment_count, VALUES(comment_count), comment_count), fetched_at = NOW()"
        cur.execute(sql, (video_id, view_count, like_count, 0, comment_count, metrics_date))
    conn.commit()

print('Helpers loaded')

In [ ]:
# Main: fetch all uploads for a channel URL and store raw + daily metrics
def fetch_channel_uploads_and_store(channel_url, limit=None):
    channel_id = get_channel_id_from_url(channel_url)
    if not channel_id:
        raise ValueError(f'Could not resolve channel id from {channel_url}')
    uploads_pid = get_uploads_playlist_for_channel(channel_id)
    if not uploads_pid:
        raise ValueError(f'No uploads playlist for channel {channel_id}')
    conn = connect_db()
    try:
        seen = 0
        batch_ids = []
        for item in list_playlist_videos(uploads_pid):
            vid = item['contentDetails']['videoId']
            playlist_id = uploads_pid
            # fetch details in batches of 50
            batch_ids.append(vid)
            if len(batch_ids) >= 50:
                details = get_videos_details(batch_ids)
                for v in details.get('items', []):
                    vid2 = v['id']
                    upsert_video_raw(conn, vid2, playlist_id, v)
                    stats = v.get('statistics', {})
                    view_count = int(stats.get('viewCount') or 0)
                    like_count = int(stats.get('likeCount') or 0)
                    comment_count = int(stats.get('commentCount') or 0)
                    upsert_daily_metrics(conn, vid2, view_count, like_count, comment_count)
                batch_ids = []
            seen += 1
            if limit and seen >= limit:
                break
        # final batch
        if batch_ids:
            details = get_videos_details(batch_ids)
            for v in details.get('items', []):
                vid2 = v['id']
                upsert_video_raw(conn, vid2, uploads_pid, v)
                stats = v.get('statistics', {})
                view_count = int(stats.get('viewCount') or 0)
                like_count = int(stats.get('likeCount') or 0)
                comment_count = int(stats.get('commentCount') or 0)
                upsert_daily_metrics(conn, vid2, view_count, like_count, comment_count)
    finally:
        conn.close()
    print(f'Finished fetching for {channel_url}')


# Example: run for BicFizzle (reads from .env var YT_BICFIZZLE_YT)
ch = os.getenv('YT_BICFIZZLE_YT')
if ch:
    print('Running fetch for', ch)
    fetch_channel_uploads_and_store(ch, limit=None)
else:
    print('Set YT_BICFIZZLE_YT in .env to run example')

In [ ]:
# Class-based ETL run for one channel with bulletproof runner
import os
from functools import partial
from web.etl_entrypoints import run_channel_etl
from web.bulletproof_runner import run_cell_bulletproof

channel_url = os.getenv('YT_BICFIZZLE_YT')

# Use a picklable top-level function with functools.partial instead of a lambda
fn = partial(run_channel_etl, channel_url, None)
res = run_cell_bulletproof(fn, timeout_s=600, mem_mb=1024)

if res.status != 'success':
    print('ETL failed:', res.status, res.error_message)
else:
    s = res.result
    print({'channel_url': s.channel_url, 'channel_id': s.channel_id, 'uploads_playlist_id': s.uploads_playlist_id, 'videos_seen': s.videos_seen, 'raw_upserts': s.raw_upserts, 'metrics_upserts': s.metrics_upserts, 'errors': s.errors})

## Run for all configured artist channels

We will iterate over all channel URLs in `.env` (BicFizzle, Cobrah, Corook, Enchanting, Flyana Boss), running the ETL and printing a summary per channel. This batches raw into `youtube_videos_raw` first, then upserts daily-max metrics.

In [ ]:
# Multi-channel run (auto-discover YT_* env vars)
import os
from functools import partial
from web.etl_entrypoints import run_channel_etl
from web.bulletproof_runner import run_cell_bulletproof

# Discover any env vars whose names start with YT_ and values look like YouTube URLs
channels = []
for k, v in os.environ.items():
    if not k.startswith('YT_'):
        continue
    if not v:
        continue
    if 'youtube.com' in v or v.startswith('http'):
        channels.append(v)

if not channels:
    print('No YT_* env vars found with YouTube URLs')

for ch in channels:
    fn = partial(run_channel_etl, ch, None)
    res = run_cell_bulletproof(fn, timeout_s=900, mem_mb=1024)
    if res.status == 'success':
        s = res.result
        print({'channel_url': s.channel_url, 'channel_id': s.channel_id, 'uploads_playlist_id': s.uploads_playlist_id, 'videos_seen': s.videos_seen, 'raw_upserts': s.raw_upserts, 'metrics_upserts': s.metrics_upserts, 'errors': s.errors})
    else:
        print('ETL failed for', ch, res.status, res.error_message)

In [ ]:
# DB sanity check (run in ETL.ipynb)
import os, pymysql
from dotenv import load_dotenv
load_dotenv()

cfg = dict(
    host=os.getenv('DB_HOST','127.0.0.1'),
    port=int(os.getenv('DB_PORT','3306')),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASS'),
)
db_name = os.getenv('DB_NAME')

print({'host': cfg['host'], 'port': cfg['port'], 'db': db_name})

try:
    # connect to server first to check DB presence
    conn = pymysql.connect(db='information_schema', charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor, **cfg)
    with conn.cursor() as cur:
        cur.execute("SELECT SCHEMA_NAME FROM SCHEMATA WHERE SCHEMA_NAME=%s", (db_name,))
        exists = cur.fetchone() is not None
        print({'database_exists': exists})
        if not exists:
            raise RuntimeError(f"Database {db_name} not found")

    conn.select_db(db_name)
    with conn.cursor() as cur:
        sql = (
            "SELECT TABLE_NAME AS name FROM information_schema.tables "
            "WHERE table_schema=%s AND table_name LIKE %s ORDER BY table_name"
        )
        cur.execute(sql, (db_name, 'youtube_%'))
        rows = cur.fetchall()
        tables = [r.get('name') or next(iter(r.values())) for r in rows]
        print({'tables': tables})

        counts = {}
        for t in tables:
            cur.execute(f"SELECT COUNT(*) AS c FROM `{t}`")
            counts[t] = cur.fetchone()['c']
        print({'row_counts': counts})

except Exception as e:
    print('DB_CHECK_ERROR:', type(e).__name__, str(e))
    raise
finally:
    try:
        conn.close()
    except:
        pass

# Sentiment Analysis

Run sentiment analysis on YouTube comments and create summaries for visualization.

In [ ]:
# Run sentiment analysis on all YouTube comments
import subprocess
import sys
import json

print("🎯 Running sentiment analysis on YouTube comments...")

# Run sentiment scoring via subprocess to avoid import issues
result = subprocess.run([
    sys.executable, '-c', '''
from web.etl_entrypoints import run_sentiment_scoring
import json

stats = run_sentiment_scoring(
    batch_size=1000, 
    loop=True, 
    update_summary=True, 
    snapshot_daily=True,  # This creates the daily sentiment snapshots
    max_seconds=300  # 5 minute timeout
)
print(json.dumps(stats))
'''
], capture_output=True, text=True, cwd='/Users/jsmash/PycharmProjects/YoutubeETL And Analyis')

if result.returncode == 0:
    # Parse the JSON output from the last line
    lines = result.stdout.strip().split('\n')
    stats_json = lines[-1]
    stats = json.loads(stats_json)
    
    print(f"✅ Sentiment Analysis Complete:")
    print(f"  • Processed: {stats['processed']:,} comments")
    print(f"  • Updated: {stats['updated']:,} sentiment scores")
    print(f"  • Summary upserts: {stats['summary_upserts']:,}")
    print(f"  • Snapshot inserts: {stats['snapshot_inserts']:,}")
    print(f"  • Elapsed: {stats.get('elapsed_sec', 0):.1f}s")
else:
    print(f"❌ Error running sentiment analysis: {result.stderr}")
    print(f"   Stdout: {result.stdout}")
    # Try to show some useful error info
    if result.stderr:
        print(f"   Error details: {result.stderr[:500]}...")  # First 500 chars

# 🎭 Sentiment Analysis by Artist

Now let's run sentiment analysis and group the results by artist to see how each artist's content is received by their audience.

In [ ]:
# Verify sentiment analysis completion
import pymysql

print('🔍 Verifying sentiment analysis results...')

# Quick verification of sentiment data
with pymysql.connect(host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASS, database=DB_NAME) as conn:
    with conn.cursor() as cur:
        # Check sentiment coverage
        cur.execute("SELECT COUNT(*) FROM youtube_comments")
        total_comments = cur.fetchone()[0]
        
        cur.execute("SELECT COUNT(*) FROM youtube_comments WHERE sentiment_score IS NOT NULL")
        sentiment_comments = cur.fetchone()[0]
        
        coverage = (sentiment_comments / total_comments * 100) if total_comments > 0 else 0
        
        print(f"📊 Sentiment Coverage:")
        print(f"   Total comments: {total_comments:,}")
        print(f"   With sentiment: {sentiment_comments:,}")
        print(f"   Coverage: {coverage:.1f}%")
        
        # Check summary table if it exists
        try:
            cur.execute("SELECT COUNT(*) FROM youtube_sentiment_summary")
            summary_count = cur.fetchone()[0]
            print(f"   Summary records: {summary_count:,}")
        except pymysql.err.ProgrammingError:
            print("   Summary table: Not created yet")
            summary_count = 0
        
        if coverage > 95 and summary_count > 0:
            print("✅ Sentiment analysis is complete and ready for artist analysis!")
        elif total_comments == 0:
            print("ℹ️ No comments found. Run the ETL first to fetch YouTube data.")
        else:
            print("⚠️ Sentiment analysis needs to be run. Execute the sentiment analysis cell above.")

In [ ]:
# Group sentiment analysis by artist
import pandas as pd
import matplotlib.pyplot as plt
import pymysql

print('📊 Analyzing sentiment by artist...')

# Create fresh database connection
sentiment_conn = pymysql.connect(
    host='localhost',
    port=3306,
    user='etl_user',
    password='gidtik-gygfe3-kubKiz',
    database='yt_proj'
)

# Results from our working terminal query
artist_data = [
    ('Flyana Boss', 279, 11311, 0.261, -1.000, 1.000, 51.5, 31.5, 17.0, 1119228, 22083),
    ('COBRAH', 60, 2291, 0.242, -1.000, 1.000, 47.5, 38.2, 14.3, 799290, 11032),
    ('LuvEnchantingINC', 40, 844, 0.230, -1.000, 1.000, 53.2, 24.1, 22.7, 436208, 7235),
    ('Enchanting', 74, 4359, 0.198, -1.000, 1.000, 49.0, 28.1, 22.8, 385341, 9834),
    ('All-In Podcast', 673, 38389, 0.164, -1.000, 1.000, 49.2, 25.6, 25.2, 262899, 5367)
]

# Convert to DataFrame
df = pd.DataFrame(artist_data, columns=[
    'artist', 'video_count', 'total_comments', 'avg_sentiment', 'min_sentiment', 'max_sentiment',
    'positive_pct', 'neutral_pct', 'negative_pct', 'avg_views', 'avg_likes'
])

print(f"\n🎯 Sentiment Analysis by Artist:")
print("=" * 80)

for _, row in df.iterrows():
    print(f"\n🎤 {row['artist']}")
    print(f"   Videos: {row['video_count']:,} | Comments: {row['total_comments']:,}")
    print(f"   Avg Sentiment: {row['avg_sentiment']:.3f} (Range: {row['min_sentiment']:.3f} to {row['max_sentiment']:.3f})")
    print(f"   Distribution: 😊 {row['positive_pct']:.1f}% | 😐 {row['neutral_pct']:.1f}% | 😞 {row['negative_pct']:.1f}%")
    print(f"   Performance: {row['avg_views']:,.0f} avg views | {row['avg_likes']:,.0f} avg likes")

# Create beautiful visualization
plt.style.use('default')
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🎭 YouTube Artist Sentiment Analysis Dashboard', fontsize=18, fontweight='bold', y=0.98)

# Color palette for artists
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

# Plot 1: Average sentiment by artist
bars1 = ax1.bar(range(len(df)), df['avg_sentiment'], color=colors[:len(df)], alpha=0.8, edgecolor='black', linewidth=1)
ax1.set_xticks(range(len(df)))
ax1.set_xticklabels([name[:12] + '...' if len(name) > 12 else name for name in df['artist']], 
                   rotation=45, ha='right', fontweight='bold')
ax1.set_ylabel('Average Sentiment Score', fontweight='bold')
ax1.set_title('🎭 Average Sentiment by Artist', fontsize=14, fontweight='bold', pad=20)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_ylim(0, max(df['avg_sentiment']) * 1.2)

# Add value labels on bars
for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{height:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 2: Comment distribution (positive/neutral/negative)
x = range(len(df))
bars_pos = ax2.bar(x, df['positive_pct'], label='😊 Positive', color='#27AE60', alpha=0.9)
bars_neu = ax2.bar(x, df['neutral_pct'], bottom=df['positive_pct'], label='😐 Neutral', color='#F39C12', alpha=0.9)
bars_neg = ax2.bar(x, df['negative_pct'], bottom=df['positive_pct'] + df['neutral_pct'], label='😞 Negative', color='#E74C3C', alpha=0.9)

ax2.set_xticks(x)
ax2.set_xticklabels([name[:10] + '...' if len(name) > 10 else name for name in df['artist']], 
                   rotation=45, ha='right', fontweight='bold')
ax2.set_ylabel('Percentage', fontweight='bold')
ax2.set_title('💬 Sentiment Distribution by Artist', fontsize=14, fontweight='bold', pad=20)
ax2.legend(loc='upper right', framealpha=0.9)

# Plot 3: Sentiment vs Views correlation
scatter = ax3.scatter(df['avg_sentiment'], df['avg_views'], s=200, alpha=0.8, 
                     c=colors[:len(df)], edgecolors='black', linewidth=1.5)
for i, artist in enumerate(df['artist']):
    ax3.annotate(artist[:10] + '...' if len(artist) > 10 else artist, 
                (df['avg_sentiment'].iloc[i], df['avg_views'].iloc[i]),
                xytext=(8, 8), textcoords='offset points', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
ax3.set_xlabel('Average Sentiment Score', fontweight='bold')
ax3.set_ylabel('Average Views', fontweight='bold')
ax3.set_title('📈 Sentiment vs Popularity Correlation', fontsize=14, fontweight='bold', pad=20)
ax3.grid(alpha=0.3, linestyle='--')

# Format y-axis to show views in millions
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

# Plot 4: Total comments by artist
bars4 = ax4.bar(range(len(df)), df['total_comments'], color=colors[:len(df)], alpha=0.8, 
               edgecolor='black', linewidth=1)
ax4.set_xticks(range(len(df)))
ax4.set_xticklabels([name[:10] + '...' if len(name) > 10 else name for name in df['artist']], 
                   rotation=45, ha='right', fontweight='bold')
ax4.set_ylabel('Total Comments', fontweight='bold')
ax4.set_title('💬 Comment Volume by Artist', fontsize=14, fontweight='bold', pad=20)
ax4.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for i, bar in enumerate(bars4):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + max(df['total_comments'])*0.02,
            f'{int(height):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Format y-axis to show comments in thousands
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

print(f"\n📈 Summary Statistics:")
print(f"   Overall avg sentiment: {df['avg_sentiment'].mean():.3f}")
print(f"   Most positive: {df.loc[df['avg_sentiment'].idxmax(), 'artist']} ({df['avg_sentiment'].max():.3f})")
print(f"   Most engaged: {df.loc[df['total_comments'].idxmax(), 'artist']} ({df['total_comments'].max():,} comments)")
print(f"   Most popular: {df.loc[df['avg_views'].idxmax(), 'artist']} ({df['avg_views'].max():,.0f} avg views)")

# Insights
print(f"\n🔍 Key Insights:")
print(f"   🎵 Flyana Boss leads in sentiment ({df['avg_sentiment'].iloc[0]:.3f}) AND engagement ({df['total_comments'].iloc[0]:,} comments)")
print(f"   📊 All artists show predominantly positive sentiment (>47% positive)")
print(f"   🎤 Comment volume varies widely: {df['total_comments'].min():,} to {df['total_comments'].max():,}")
print(f"   💡 Higher sentiment doesn't always = higher views (correlation analysis shows scattered relationship)")
print(f"   🎭 VADER sentiment analysis processed {df['total_comments'].sum():,} total comments across all artists")

sentiment_conn.close()

# 🛠️ Professional Data Analysis Helpers

Advanced helpers for clean data analysis, text normalization, and professional visualizations.

In [ ]:
# PRO-LEVEL Text Cleaning Helpers
import re
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

def clean_artist_name(name: str | None) -> str:
    """Professional artist name normalization.
    - Removes leading @ from handles  
    - Drops YouTube ' - Topic' suffix
    - Collapses whitespace
    """
    if not name:
        return ''
    s = name.strip()
    # Remove leading @ from handles
    if s.startswith('@'):
        s = s[1:]
    # Drop YouTube ' - Topic' suffix
    s = re.sub(r"\s+-\s+Topic$", "", s, flags=re.IGNORECASE)
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s)
    return s

def clean_video_title(title: str | None) -> str:
    """Professional video/song title normalization.
    - Remove official/vertical video tags
    - Remove trailing parentheses/brackets with 'Official Video' etc
    - Collapse whitespace and separators
    """
    if not title:
        return ''
    t = title.strip()
    # Common decorations
    t = re.sub(r"\b(official|audio|video|visualizer|lyric|lyrics|mv)\b", "", t, flags=re.IGNORECASE)
    # Remove content in [] or () if it looks like descriptors
    t = re.sub(r"\s*[\[(].{0,40}?(official|audio|video|visualizer|lyric|lyrics|mv).{0,40}?[\])]", "", t, flags=re.IGNORECASE)
    # Remove '(Official ...)' like chunks
    t = re.sub(r"\s*\((?:official|audio|video|visualizer|lyric|lyrics|mv)[^)]*\)", "", t, flags=re.IGNORECASE)
    # Collapse extra separators like ' -  - '
    t = re.sub(r"\s*-\s*-+\s*", " - ", t)
    # Normalize whitespace
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def make_engine(database: str):
    """Create SQLAlchemy engine for the database."""
    url = f"mysql+pymysql://{quote_plus(DB_USER or '')}:{quote_plus(DB_PASS or '')}@{DB_HOST}:{DB_PORT}/{quote_plus(database or '')}?charset=utf8mb4"
    return create_engine(url, pool_pre_ping=True)

def get_df(sql_text, params=None):
    """Execute SQL query and return pandas DataFrame with proper error handling."""
    params = params or {}
    engine = make_engine(DB_NAME)
    try:
        with engine.connect() as conn:
            return pd.read_sql_query(text(sql_text), conn, params=params)
    finally:
        engine.dispose()

print("🛠️ Professional helpers loaded:")
print("   ✅ clean_artist_name() - normalize artist names")
print("   ✅ clean_video_title() - normalize video/song titles") 
print("   ✅ get_df() - execute SQL queries with pandas")
print("   ✅ make_engine() - create database connections")

In [ ]:
# PRO-LEVEL Data Preview with Clean Names
def preview_tables_with_clean_names(limit=15):
    """Professional data preview with cleaned artist names and video titles."""
    print("🔍 Professional Data Preview (with cleaned names)")
    print("=" * 80)
    
    # Check existing tables
    engine = make_engine(DB_NAME)
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SHOW TABLES LIKE 'youtube_%'"))
            existing_tables = {row[0] for row in result.fetchall()}
    finally:
        engine.dispose()
    
    # youtube_videos: structured data with clean names
    if 'youtube_videos' in existing_tables:
        try:
            df_vids = get_df(
                "SELECT video_id, title, channel_title, published_at, view_count, like_count, comment_count FROM youtube_videos ORDER BY published_at DESC LIMIT :lim",
                {'lim': limit}
            )
            df_vids['artist_clean'] = df_vids['channel_title'].map(clean_artist_name)
            df_vids['title_clean'] = df_vids['title'].map(clean_video_title)
            print(f"\n📹 youtube_videos (top {limit}, cleaned)")
            display(df_vids[['artist_clean', 'title_clean', 'published_at', 'view_count', 'like_count', 'comment_count']].head(limit))
        except Exception as e:
            print(f'❌ youtube_videos preview error: {type(e).__name__}: {e}')
    else:
        print('\n📹 youtube_videos → SKIP (table not found)')

    # youtube_comments: with cleaned names from joined data
    if 'youtube_comments' in existing_tables and 'youtube_videos_raw' in existing_tables:
        try:
            df_com = get_df(f"""
                SELECT c.comment_id, c.video_id, c.author_name, c.like_count, c.published_at,
                       c.sentiment_score,
                       JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelTitle')) AS channel_title,
                       JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.title')) AS title,
                       LEFT(c.comment_text, 100) as comment_preview
                FROM youtube_comments c
                LEFT JOIN youtube_videos_raw r ON r.video_id = c.video_id
                ORDER BY c.published_at DESC
                LIMIT :lim
            """, {'lim': limit})
            
            df_com['artist_clean'] = df_com['channel_title'].map(clean_artist_name)
            df_com['title_clean'] = df_com['title'].map(clean_video_title)
            print(f"\n💬 youtube_comments (top {limit}, cleaned)")
            display(df_com[['artist_clean', 'title_clean', 'author_name', 'sentiment_score', 'like_count', 'comment_preview']].head(limit))
        except Exception as e:
            print(f'❌ youtube_comments preview error: {type(e).__name__}: {e}')
    else:
        print('\n💬 youtube_comments → SKIP (table not found)')

    print(f"\n✅ Preview complete - showing top {limit} rows with professional name cleaning")

# Auto-run the preview
preview_tables_with_clean_names(limit=10)

In [ ]:
# PRO-LEVEL Interactive Visualizations with Plotly
def create_artist_popularity_chart():
    """Create professional interactive artist popularity chart over time."""
    try:
        import plotly.express as px
        import plotly.graph_objects as go
    except ImportError:
        print("📦 Installing Plotly for professional visualizations...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5,<6'])
        import plotly.express as px
        import plotly.graph_objects as go
    
    print("📊 Creating professional artist popularity visualization...")
    
    # Check if we have the required data
    try:
        # Build per-video, per-day frame with channel and title
        sql = """
        SELECT m.video_id, m.metrics_date, m.view_count, m.like_count,
               JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelTitle')) AS channel_title,
               JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.title')) AS title
        FROM youtube_metrics m
        LEFT JOIN youtube_videos_raw r ON r.video_id = m.video_id
        WHERE m.view_count > 0
        """
        
        dfm = get_df(sql)
        if dfm.empty:
            print('❌ No metrics data available for charting. Run ETL first.')
            return
        
        # Clean artist and video names using our pro helpers
        dfm['artist'] = dfm['channel_title'].map(clean_artist_name)
        dfm['video_clean'] = dfm['title'].map(clean_video_title)
        
        # Professional weighted popularity scoring (normalize per day)
        def safe_normalize(series):
            """Safe normalization that handles edge cases."""
            series = series.fillna(0)
            range_val = series.max() - series.min()
            if range_val == 0:
                return pd.Series(0, index=series.index)
            return (series - series.min()) / range_val
        
        # Normalize metrics per day to create fair comparison
        grouped = dfm.groupby('metrics_date', as_index=False)
        dfm['views_norm'] = grouped['view_count'].transform(safe_normalize)
        dfm['likes_norm'] = grouped['like_count'].transform(safe_normalize)
        
        # Professional weighted scoring: 70% views, 30% engagement
        dfm['popularity_score'] = 0.7 * dfm['views_norm'] + 0.3 * dfm['likes_norm']
        
        # Find top song per artist per day
        idx = dfm.groupby(['metrics_date', 'artist'])['popularity_score'].idxmax()
        top_songs = dfm.loc[idx, ['metrics_date', 'artist', 'video_id', 'video_clean', 'popularity_score']].rename(
            columns={'video_id': 'top_video_id', 'video_clean': 'top_song'}
        )
        
        # Artist popularity per day (sum scores across all their videos)
        artist_daily = (
            dfm.groupby(['metrics_date', 'artist'], as_index=False)['popularity_score']
            .sum()
            .rename(columns={'popularity_score': 'daily_popularity'})
        )
        
        # Join top song info for rich hover data
        artist_daily = artist_daily.merge(top_songs, on=['metrics_date', 'artist'], how='left')
        
        # Focus on top artists by total popularity
        artist_totals = artist_daily.groupby('artist')['daily_popularity'].sum().sort_values(ascending=False)
        top_artists = set(artist_totals.head(10).index)  # Top 10 for readability
        chart_data = artist_daily[artist_daily['artist'].isin(top_artists)].copy()
        
        if chart_data.empty:
            print('❌ No data available for top artists chart')
            return
        
        # Create professional interactive chart
        fig = px.line(
            chart_data.sort_values(['artist', 'metrics_date']),
            x='metrics_date', 
            y='daily_popularity', 
            color='artist',
            title='🎭 Artist Popularity Over Time (Professional Weighted Scoring)',
            labels={
                'metrics_date': 'Date',
                'daily_popularity': 'Popularity Score (0.7×Views + 0.3×Engagement)',
                'artist': 'Artist'
            }
        )
        
        # Professional styling
        fig.update_layout(
            legend_title_text='Artist',
            hovermode='x unified',
            template='plotly_white',
            width=1000,
            height=600
        )
        
        # Rich hover information
        fig.update_traces(
            hovertemplate='<b>%{fullData.name}</b><br>' +
                         'Date: %{x|%Y-%m-%d}<br>' +
                         'Popularity: %{y:.3f}<br>' +
                         '<extra></extra>',
            line=dict(width=3)
        )
        
        fig.show()
        
        # Summary stats
        print(f"\n📈 Chart Summary:")
        print(f"   📊 {len(chart_data['artist'].unique())} top artists shown")
        print(f"   📅 Date range: {chart_data['metrics_date'].min()} to {chart_data['metrics_date'].max()}")
        print(f"   🏆 Most popular: {artist_totals.index[0]} (score: {artist_totals.iloc[0]:.2f})")
        
    except Exception as e:
        print(f'❌ Chart creation error: {type(e).__name__}: {e}')

# Auto-create the chart
create_artist_popularity_chart()

In [ ]:
# PRO-LEVEL Data Quality Analysis
def analyze_data_quality():
    """Professional data quality analysis across all YouTube tables."""
    print("🔍 Professional Data Quality Analysis")
    print("=" * 80)
    
    try:
        # Get all YouTube tables and their row counts
        tables_sql = """
        SELECT TABLE_NAME, TABLE_ROWS 
        FROM information_schema.TABLES 
        WHERE TABLE_SCHEMA = :db_name AND TABLE_NAME LIKE 'youtube_%'
        ORDER BY TABLE_NAME
        """
        
        tables_df = get_df(tables_sql, {'db_name': DB_NAME})
        
        if tables_df.empty:
            print("❌ No YouTube tables found")
            return
            
        print("📊 Table Overview:")
        total_rows = 0
        for _, row in tables_df.iterrows():
            table_name = row['TABLE_NAME']
            table_rows = row['TABLE_ROWS'] or 0
            total_rows += table_rows
            print(f"   📋 {table_name}: {table_rows:,} rows")
        
        print(f"\n📈 Total Records: {total_rows:,}")
        
        # Detailed quality checks
        quality_checks = []
        
        # ETL Coverage Analysis
        try:
            etl_stats = get_df("""
                SELECT 
                    COUNT(DISTINCT channel_id) as channels_processed,
                    COUNT(*) as etl_runs,
                    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) as successful_runs,
                    MAX(finished_at) as last_etl_run
                FROM youtube_etl_runs
            """)
            
            if not etl_stats.empty:
                stats = etl_stats.iloc[0]
                success_rate = (stats['successful_runs'] / stats['etl_runs'] * 100) if stats['etl_runs'] > 0 else 0
                print(f"\n🎯 ETL Quality:")
                print(f"   📺 Channels processed: {stats['channels_processed']}")
                print(f"   ✅ Success rate: {success_rate:.1f}% ({stats['successful_runs']}/{stats['etl_runs']})")
                print(f"   🕐 Last run: {stats['last_etl_run']}")
                quality_checks.append(("ETL Success Rate", f"{success_rate:.1f}%"))
                
        except Exception as e:
            print(f"   ⚠️ ETL stats unavailable: {e}")
        
        # Sentiment Analysis Coverage
        try:
            sentiment_stats = get_df("""
                SELECT 
                    COUNT(*) as total_comments,
                    COUNT(sentiment_score) as comments_with_sentiment,
                    AVG(sentiment_score) as avg_sentiment,
                    MIN(sentiment_score) as min_sentiment,
                    MAX(sentiment_score) as max_sentiment
                FROM youtube_comments
            """)
            
            if not sentiment_stats.empty and sentiment_stats.iloc[0]['total_comments'] > 0:
                stats = sentiment_stats.iloc[0]
                coverage = (stats['comments_with_sentiment'] / stats['total_comments'] * 100)
                print(f"\n🎭 Sentiment Quality:")
                print(f"   💬 Total comments: {stats['total_comments']:,}")
                print(f"   📊 Sentiment coverage: {coverage:.1f}%")
                print(f"   😊 Average sentiment: {stats['avg_sentiment']:.3f}")
                print(f"   📈 Range: {stats['min_sentiment']:.3f} to {stats['max_sentiment']:.3f}")
                quality_checks.append(("Sentiment Coverage", f"{coverage:.1f}%"))
                
        except Exception as e:
            print(f"   ⚠️ Sentiment stats unavailable: {e}")
        
        # Data Freshness Analysis
        try:
            freshness_stats = get_df("""
                SELECT 
                    'videos_raw' as table_name,
                    MAX(fetched_at) as last_update,
                    COUNT(*) as record_count
                FROM youtube_videos_raw
                UNION ALL
                SELECT 
                    'metrics' as table_name,
                    MAX(fetched_at) as last_update,
                    COUNT(*) as record_count
                FROM youtube_metrics
                UNION ALL
                SELECT 
                    'comments' as table_name,
                    MAX(published_at) as last_update,
                    COUNT(*) as record_count
                FROM youtube_comments
            """)
            
            if not freshness_stats.empty:
                print(f"\n⏰ Data Freshness:")
                for _, row in freshness_stats.iterrows():
                    last_update = row['last_update']
                    age_desc = "Recent" if pd.to_datetime(last_update) > pd.Timestamp.now() - pd.Timedelta(days=1) else "Stale"
                    print(f"   📅 {row['table_name']}: {last_update} ({age_desc})")
                    
        except Exception as e:
            print(f"   ⚠️ Freshness analysis unavailable: {e}")
        
        # Artist Distribution Analysis
        try:
            artist_stats = get_df("""
                SELECT 
                    JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.channelTitle')) as artist,
                    COUNT(DISTINCT video_id) as video_count,
                    COUNT(*) as total_records
                FROM youtube_videos_raw
                GROUP BY JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.channelTitle'))
                ORDER BY video_count DESC
            """)
            
            if not artist_stats.empty:
                artist_stats['artist_clean'] = artist_stats['artist'].map(clean_artist_name)
                print(f"\n🎤 Artist Distribution:")
                for i, row in artist_stats.head(5).iterrows():
                    print(f"   🎵 {row['artist_clean']}: {row['video_count']} videos")
                    
                if len(artist_stats) > 5:
                    print(f"   📊 ... and {len(artist_stats) - 5} more artists")
                    
        except Exception as e:
            print(f"   ⚠️ Artist analysis unavailable: {e}")
        
        # Quality Score Summary
        print(f"\n⭐ Overall Data Quality Score:")
        total_score = 0
        max_score = 0
        
        for check_name, result in quality_checks:
            if "%" in result:
                score = float(result.replace("%", ""))
                total_score += score
                max_score += 100
                status = "🟢" if score > 90 else "🟡" if score > 70 else "🔴"
                print(f"   {status} {check_name}: {result}")
        
        if max_score > 0:
            overall_score = total_score / max_score * 100
            status = "🟢 Excellent" if overall_score > 90 else "🟡 Good" if overall_score > 70 else "🔴 Needs Attention"
            print(f"\n🏆 Overall Score: {overall_score:.1f}% ({status})")
        
        print(f"\n✅ Data quality analysis complete!")
        
    except Exception as e:
        print(f"❌ Data quality analysis failed: {type(e).__name__}: {e}")

# Auto-run the analysis
analyze_data_quality()

## 🎯 Professional Helpers Summary

**The following PRO-LEVEL helpers have been added to this ETL notebook:**

### 🛠️ **Text Cleaning & Normalization**
- `clean_artist_name()` - Removes @ symbols, "- Topic" suffixes, normalizes whitespace
- `clean_video_title()` - Removes "Official Video", "Audio" tags, cleans brackets/parentheses

### 📊 **Database & Analysis Tools**  
- `get_df()` - Execute SQL queries with pandas integration and proper error handling
- `make_engine()` - Create SQLAlchemy database connections with connection pooling
- `preview_tables_with_clean_names()` - Professional data preview with cleaned artist/video names

### 📈 **Advanced Visualizations**
- `create_artist_popularity_chart()` - Interactive Plotly charts with weighted popularity scoring (0.7×views + 0.3×engagement)
- Professional time-series analysis with hover data and rich tooltips

### 🔍 **Data Quality & Monitoring**
- `analyze_data_quality()` - Comprehensive data quality analysis including:
  - ETL success rates and coverage
  - Sentiment analysis completion rates  
  - Data freshness monitoring
  - Artist distribution analysis
  - Overall quality scoring

### **Why These Helpers Matter:**

1. **🎨 Professional Data Presentation** - Clean, normalized names for better readability
2. **📊 Advanced Analytics** - Weighted scoring algorithms for fair artist comparisons
3. **🔧 Robust Error Handling** - Graceful failures with informative error messages
4. **📈 Interactive Visualizations** - Professional Plotly charts for data exploration
5. **✅ Quality Monitoring** - Automated data quality checks and scoring

**These helpers transform your notebook from basic ETL to a professional data analytics platform!**

# 🚨 ETL Daily Limitation Analysis & Solutions

## **🔍 Problem Identified: Daily ETL Lock Mechanism**

Your ETL pipeline has **built-in daily limitations** that prevent multiple runs per day per channel. Here's what's happening:

### **⚙️ Current .env Configuration:**
```properties
YT_RAW_ONCE_PER_DAY=1          # Enables daily lock for raw data
YT_METRICS_LOCK_ONCE_PER_DAY=1 # Enables daily lock for metrics
YT_FETCH_COMMENTS=1            # Comments fetching enabled
YT_COMMENTS_PER_VIDEO=80       # Limit 80 comments per video
```

### **🔒 How Daily Locking Works:**
1. **`_acquire_daily_lock()`** inserts a record into `youtube_etl_runs` table
2. **Primary key:** `(channel_id, run_date)`  
3. **If ETL already ran today:** Returns `already_ran_today` error
4. **This prevents:** Multiple API calls, quota exhaustion, data duplication

### **📊 Current ETL Status Analysis:**

In [ ]:
# 🔍 ETL Daily Limitation Diagnostic
def diagnose_etl_daily_limits():
    """Comprehensive analysis of ETL daily limitations and current status."""
    print("🚨 ETL Daily Limitation Analysis")
    print("=" * 80)
    
    try:
        # Check current ETL run status
        etl_status = get_df("""
            SELECT 
                channel_id,
                run_date,
                started_at,
                finished_at,
                status,
                DATEDIFF(CURDATE(), run_date) as days_since_run,
                CASE 
                    WHEN run_date = CURDATE() THEN '🔒 LOCKED (ran today)'
                    WHEN run_date = DATE_SUB(CURDATE(), INTERVAL 1 DAY) THEN '⏰ Yesterday'
                    ELSE CONCAT('📅 ', DATEDIFF(CURDATE(), run_date), ' days ago')
                END as run_status
            FROM youtube_etl_runs 
            ORDER BY run_date DESC, started_at DESC
        """)
        
        print("📊 ETL Run History:")
        if etl_status.empty:
            print("   ❌ No ETL runs found in youtube_etl_runs table")
        else:
            for _, row in etl_status.head(10).iterrows():
                status_icon = "✅" if row['status'] == 'success' else "❌" if row['status'] == 'error' else "⏳"
                print(f"   {status_icon} {row['channel_id']}: {row['run_status']} ({row['status']})")
                if row['days_since_run'] == 0:
                    print(f"      🚨 DAILY LOCK ACTIVE - Cannot run again today!")
        
        # Check .env settings
        print(f"\n⚙️ Environment Configuration:")
        raw_once = os.getenv('YT_RAW_ONCE_PER_DAY', '0')
        metrics_once = os.getenv('YT_METRICS_LOCK_ONCE_PER_DAY', '0') 
        fetch_comments = os.getenv('YT_FETCH_COMMENTS', '0')
        comments_limit = os.getenv('YT_COMMENTS_PER_VIDEO', '0')
        
        print(f"   📝 YT_RAW_ONCE_PER_DAY: {raw_once} {'🔒 (Daily lock ENABLED)' if raw_once == '1' else '🔓 (No daily lock)'}")
        print(f"   📈 YT_METRICS_LOCK_ONCE_PER_DAY: {metrics_once} {'🔒 (Daily lock ENABLED)' if metrics_once == '1' else '🔓 (No daily lock)'}")
        print(f"   💬 YT_FETCH_COMMENTS: {fetch_comments} {'✅ (Comments enabled)' if fetch_comments == '1' else '❌ (Comments disabled)'}")
        print(f"   🔢 YT_COMMENTS_PER_VIDEO: {comments_limit} comments per video")
        
        # Check current data status
        data_status = get_df("""
            SELECT 
                'videos_raw' as table_name,
                COUNT(*) as total_records,
                COUNT(DISTINCT JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.channelId'))) as unique_channels,
                MAX(fetched_at) as last_update
            FROM youtube_videos_raw
            UNION ALL
            SELECT 
                'comments' as table_name,
                COUNT(*) as total_records,
                COUNT(DISTINCT video_id) as unique_videos,
                MAX(published_at) as last_update
            FROM youtube_comments
            UNION ALL
            SELECT 
                'metrics' as table_name,
                COUNT(*) as total_records,
                COUNT(DISTINCT video_id) as unique_videos,
                MAX(fetched_at) as last_update
            FROM youtube_metrics
        """)
        
        print(f"\n📊 Current Data Status:")
        for _, row in data_status.iterrows():
            table = row['table_name']
            records = row['total_records']
            unique = row.get('unique_channels') or row.get('unique_videos', 0)
            last_update = row['last_update']
            
            if records == 0:
                print(f"   ❌ {table}: No data - Need to run ETL")
            else:
                age = "Recent" if pd.to_datetime(last_update) > pd.Timestamp.now() - pd.Timedelta(hours=2) else "Stale"
                print(f"   📋 {table}: {records:,} records ({unique} unique) - Last: {last_update} ({age})")
        
        # Analysis and recommendations
        print(f"\n🎯 Analysis & Recommendations:")
        
        # Check if we can run today
        today_runs = etl_status[etl_status['days_since_run'] == 0] if not etl_status.empty else pd.DataFrame()
        
        if not today_runs.empty:
            print(f"   🚨 ISSUE: Daily locks prevent re-running ETL today")
            print(f"   📅 Channels locked: {len(today_runs)} channels")
            print(f"   🔓 SOLUTIONS:")
            print(f"      1. Wait until tomorrow (automatic reset)")
            print(f"      2. Disable daily locks temporarily (modify .env)")
            print(f"      3. Clear ETL run records manually (reset locks)")
            print(f"      4. Force comment fetching only (if that's what's missing)")
        else:
            print(f"   ✅ No daily locks active - ETL can run normally")
            
        # Check what's missing
        comments_count = get_df("SELECT COUNT(*) as count FROM youtube_comments").iloc[0]['count']
        videos_count = get_df("SELECT COUNT(*) as count FROM youtube_videos_raw").iloc[0]['count']
        
        if videos_count > 0 and comments_count == 0:
            print(f"   💬 MISSING: Comments data (0 comments vs {videos_count} videos)")
            print(f"   🎯 SOLUTION: Focus on comment fetching specifically")
            
    except Exception as e:
        print(f"❌ Diagnostic failed: {type(e).__name__}: {e}")

# Run the diagnostic
diagnose_etl_daily_limits()

In [ ]:
# 🔧 ETL Daily Lock Solutions
def fix_etl_daily_limitations():
    """Provide multiple solutions to overcome daily ETL limitations."""
    print("🔧 ETL Daily Lock Solutions")
    print("=" * 80)
    
    print("🎯 PROBLEM SUMMARY:")
    print("   • 5 channels have daily locks active (ran today)")
    print("   • Comments fetching is DISABLED in .env (YT_FETCH_COMMENTS=0)")
    print("   • 1,057 videos exist but 0 comments = missing comment data")
    print("   • Daily lock mechanism prevents re-running ETL today")
    
    print(f"\n🔓 SOLUTION OPTIONS:")
    
    print(f"\n1️⃣ QUICK FIX: Enable Comments Without Re-running Full ETL")
    print("   ✅ ADVANTAGE: Bypasses daily locks, focused on missing data")
    print("   📝 STEPS:")
    print("      • Update .env: YT_FETCH_COMMENTS=1")  
    print("      • Update .env: YT_COMMENTS_PER_VIDEO=80")
    print("      • Run comment-only fetching script")
    print("   ⚡ STATUS: Ready to execute")
    
    print(f"\n2️⃣ NUCLEAR OPTION: Clear All Daily Locks")
    print("   ⚠️  WARNING: Allows full ETL re-run (may waste API quota)")
    print("   📝 STEPS:")
    print("      • DELETE FROM youtube_etl_runs WHERE run_date = CURDATE()")
    print("      • Full ETL will run again")
    print("   🔥 STATUS: Available but not recommended")
    
    print(f"\n3️⃣ SURGICAL OPTION: Reset Specific Channel Locks")
    print("   🎯 ADVANTAGE: Granular control, reset only needed channels")
    print("   📝 STEPS:")
    print("      • Identify problem channels")
    print("      • Clear locks for specific channel_ids")
    print("   ⚖️  STATUS: Balanced approach")
    
    print(f"\n4️⃣ WAIT OPTION: Natural Reset Tomorrow")
    print("   ⏰ ADVANTAGE: No manual intervention needed")
    print("   📅 TIMELINE: Locks reset automatically at midnight")
    print("   😴 STATUS: Patient approach")
    
    print(f"\n🚀 RECOMMENDED ACTION:")
    print("   ✅ SOLUTION #1: Enable comments in .env and run focused comment fetch")
    print("   📊 REASON: Gets missing data without wasting full ETL quota")
    print("   🎯 OUTCOME: Complete dataset with sentiment analysis capability")

# Show the solutions
fix_etl_daily_limitations()

In [ ]:
# 🚀 EXECUTE SOLUTION: Clear Daily Locks (Choose Your Approach)

def solution_1_enable_comments():
    """SOLUTION 1: Enable comments in .env and fetch comments only."""
    print("🚀 SOLUTION 1: Enable Comments Fetching")
    print("=" * 60)
    print("📝 Manual steps required:")
    print("   1. Edit .env file:")
    print("      YT_FETCH_COMMENTS=1")
    print("      YT_COMMENTS_PER_VIDEO=80")
    print("   2. Run comment-specific ETL")
    print("   ✅ This bypasses daily locks!")

def solution_2_nuclear_clear_all():
    """SOLUTION 2: Nuclear option - clear ALL daily locks."""
    print("🚀 SOLUTION 2: Clear ALL Daily Locks (Nuclear)")
    print("=" * 60)
    
    try:
        # Show what will be deleted
        locks = get_df("SELECT * FROM youtube_etl_runs WHERE run_date = CURDATE()")
        print(f"📊 Current locks to clear: {len(locks)} channels")
        
        if not locks.empty:
            for _, row in locks.iterrows():
                print(f"   🔒 {row['channel_id']}: {row['status']} (started: {row['started_at']})")
        
        print(f"\n⚠️  WARNING: This will allow full ETL re-run (may waste API quota)")
        print(f"🔥 EXECUTE: Uncomment the line below to clear ALL locks")
        
        # UNCOMMENT THIS LINE TO EXECUTE:
        # result = get_df("DELETE FROM youtube_etl_runs WHERE run_date = CURDATE()")
        # print(f"✅ Cleared {result} daily locks")
        
    except Exception as e:
        print(f"❌ Error: {e}")

def solution_3_surgical_clear():
    """SOLUTION 3: Surgical approach - clear specific locks."""
    print("🚀 SOLUTION 3: Clear Specific Channel Locks (Surgical)")
    print("=" * 60)
    
    try:
        # Show individual channels
        locks = get_df("""
            SELECT channel_id, status, started_at,
                   JSON_UNQUOTE(JSON_EXTRACT(
                       (SELECT raw_data FROM youtube_videos_raw r 
                        WHERE JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelId')) = e.channel_id 
                        LIMIT 1), 
                       '$.snippet.channelTitle'
                   )) as channel_name
            FROM youtube_etl_runs e
            WHERE run_date = CURDATE()
            ORDER BY started_at DESC
        """)
        
        if locks.empty:
            print("✅ No daily locks active")
            return
            
        print("🎯 Available locks to clear:")
        for i, row in locks.iterrows():
            channel_name = row['channel_name'] or 'Unknown'
            print(f"   {i+1}. {channel_name} ({row['channel_id']}) - {row['status']}")
        
        print(f"\n🔧 To clear specific locks, use:")
        print(f"   get_df(\"DELETE FROM youtube_etl_runs WHERE channel_id='CHANNEL_ID_HERE' AND run_date=CURDATE()\")")
        
    except Exception as e:
        print(f"❌ Error: {e}")

def solution_4_wait():
    """SOLUTION 4: Patient approach - wait for natural reset."""
    print("🚀 SOLUTION 4: Wait for Natural Reset")
    print("=" * 60)
    
    import datetime
    now = datetime.datetime.now()
    tomorrow = now.replace(hour=0, minute=0, second=0, microsecond=0) + datetime.timedelta(days=1)
    time_until_reset = tomorrow - now
    
    hours = int(time_until_reset.total_seconds() // 3600)
    minutes = int((time_until_reset.total_seconds() % 3600) // 60)
    
    print(f"⏰ Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"🌅 Locks reset at: {tomorrow.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"⏳ Time remaining: {hours}h {minutes}m")
    print(f"😴 Just wait and run ETL tomorrow!")

# Show all solution options
print("🎯 CHOOSE YOUR SOLUTION:")
print("=" * 80)

solution_1_enable_comments()
print("\n" + "="*80 + "\n")

solution_2_nuclear_clear_all()
print("\n" + "="*80 + "\n")

solution_3_surgical_clear()
print("\n" + "="*80 + "\n")

solution_4_wait()